# Polynomial Regression
We used multi-linear regression when we had multiple variables, but what if we had only one variable and a simple linear regression wasn't enough?
We can model $$y$$ as a polynomial function of $$x$$:

$$y\approx\beta_{0}+\beta_{1}x+\beta_{2}x^{2}+\dots+\beta_{d}x^d$$

since this may perform better than linear regression.

Due to its similarity to multi-linear regression, we can simply redefine $$x_{i}\triangleq x^{i}$$ for $$i = 1,\dots, d$$
* Same idea can be used for other nonlinear models, an example of this could be: $$x_{i}\triangleq cos(w_{i}x)$$
* This is another instance of feature transformation like one-hot coding

#### But how do we choose the polynomial order $$d$$ and why is this so important?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn

## Polynomial Regression and Model Order Selection

Going back to the multi-linear regression, we gave the following definition to our model target y:

$$y \approx \beta_{0} + \beta_{1}x_{1} + \dots + \beta_{d}x_{d}$$

Where:
* model parameters are $$\mathbf{\beta}$$ = $$[\beta_{0}, \beta_{1}, \dots, \beta_{d}]^{⊤}$$
* d is the degree of the polynomial
* given d we can fit $$\mathbf{\beta}$$ using polynomial regression

How do we select d from the data?

Below is a visualization of our data we will use for the rest of the notebook:

In [ ]:
rng = np.random.default_rng(seed=5)

beta = np.array([1,0.5,0,2])
wstd = 0.4
dtrue = len(beta)-1

nsamp = 70
xdat = rng.uniform(-1,1,nsamp)

y0 = np.polynomial.polynomial.polyval(xdat, beta)
ydat = y0 + rng.normal(0, wstd, nsamp)

plt.scatter(xdat, ydat)
plt.grid()
plt.xlabel('x')
plt.ylabel('y')

In [ ]:
beta_underfit = np.polynomial.polynomial.polyfit(xdat, ydat, 1)
beta_hat = np.polynomial.polynomial.polyfit(xdat, ydat, 3)
beta_overfit = np.polynomial.polynomial.polyfit(xdat, ydat, 15)


xp = np.linspace(-1,1,100)
yp = np.polynomial.polynomial.polyval(xp, beta)
yp_underfit = np.polynomial.polynomial.polyval(xp, beta_underfit)
yp_hat = np.polynomial.polynomial.polyval(xp, beta_hat)
yp_overfit = np.polynomial.polynomial.polyval(xp, beta_overfit)

Below is a visualization of when we use polynomial order, $$d$$, of 1 (which is the same as a simple linear regression). We don't have a very good match to the data's structure.

In [ ]:
fig, ax = plt.subplots(1,2)

fig.suptitle('Underfit V. Correct d')

ax[0].scatter(xdat, ydat)
ax[0].plot(xp,yp_underfit, 'g-', linewidth=2, label='Predicted (d=1)')
ax[0].plot(xp,yp, 'r-', linewidth=2, label='True (dtrue=3)')
ax[0].grid()
ax[0].legend()

ax[1].scatter(xdat, ydat)
ax[1].plot(xp,yp_hat, 'g-', linewidth=2, label='Predicted (d=3)')
ax[1].plot(xp,yp, 'r-', linewidth=2, label='True (dtrue=3)')
ax[1].grid()
ax[1].legend()

Below is a visualization of when we use a polynomial order of 15. We are overfitting the model to the data.

In [ ]:
fig, ax = plt.subplots(1,2)

fig.suptitle('Overfit V. Correct d')

ax[0].scatter(xdat, ydat)
ax[0].plot(xp,yp_overfit, 'g-', linewidth=2, label='Predicted (d=15)')
ax[0].plot(xp,yp, 'r-', linewidth=2, label='True (dtrue=3)')
ax[0].grid()
ax[0].legend()

ax[1].scatter(xdat, ydat)
ax[1].plot(xp,yp_hat, 'g-', linewidth=2, label='Predicted (d=3)')
ax[1].plot(xp,yp, 'r-', linewidth=2, label='True (dtrue=3)')
ax[1].grid()
ax[1].legend()

## How do we estimate the true $$d$$ from the data?
We can compute the standardized mean-squared error (sMSE) right?
* $$\frac{1}{n}\sum_{i=1}^{n}(y_{i}-\hat{y}_{i})^2=\frac{1}{n}||\mathbf{y}-\mathbf{A\hat{\beta}}||^2$$

Then we can choose $$d$$ that minimizes the sMSE. 

#### MINIMAL LOSS = BEST MODEL
Right? Unfortunately, it is not that simple. $$sMSE(d)$$ will always decrease as we increase $$d$$. Which will suggest us choose the largest $$d$$ possible. This is the issue of overfitting, but how does this occur?

We can represent what we are choosing from can be represented as the common analogy of russian nesting dolls.
* $$\mathbf{\beta}=[\beta_{0},\beta_{1}, 0, 0, \dots]$$
* $$\mathbf{\beta}=[\beta_{0},\beta_{1}, \beta_{2}, 0, \dots]$$
* $$\mathbf{\beta}=[\beta_{0},\beta_{1}, \beta_{2}, \beta_{3}, \dots]$$

* $$\vdots$$

So each model is more capable than the previous model of $$d-1$$. But as we increase $$d$$, $$\hat{y}_{i}$$ will try to fit to the noise in the data. Therefore $$d >= n-1$$ will result in a $$sMSE_{train}(d)=0$$.

### What is the solution?
Simply put we can split our data into two subsets (folds):
* Training samples
* Test samples

This will allow us to evaluate performance on test data that is independent of the training data allowing us to determine whether the noise from the training data in combination with our choice of $$d$$ is leading to overfitting. For each model-order $$d$$ we can compute the $$sMSE_{test}(d)$$ to choose the $$d$$ that minimizes this.

#### How is this any different to what we were discussing earlier?
Overfitting is less of an issue as it is harder for training noise to fit to test noise. This doesn't lead to the perfect polynomial order but it will help us narrow in on a better $$d$$.

## Where is this leading?
We talked about splitting the data into training & test samples but we can take a more complex approach to this: **K-fold Cross Validation**.

### What is K-fold cross validation?
Partitioning (shuffled) data into **K** # of folds and training using K-1 folds, and testing against 1 fold. Repeating for each of K possible test folds. The downside of this, is that it is very expensive but it is a good approximation of true performance.

This can be taken to an extreme $$K = n$$ where is the total number of samples, so you only test against one sample. This should almost never be used, might be applicable where we have small number of samples.

### How do we implement K-fold cross-validation with sklearn?

In [ ]:
k = 5
kfo = sklearn.model_selection.KFold(n_splits=k, shuffle=True)

d_test = np.arange(0,10) # model orders to be tested
n_d = len(d_test)

sMSEcv = np.zeros((n_d, k))

for isplit, Ind in enumerate(kfo.split(xdat)):
    Itr, Its = Ind
    
    xtr = xdat[Itr]
    ytr = ydat[Itr]
    xts = xdat[Its]
    yts = ydat[Its]

    for it, d in enumerate(d_test):
        beta_hat = np.polynomial.polynomial.polyfit(xtr,ytr,d)

        yhat = np.polynomial.polynomial.polyval(xts, beta_hat)
        sMSEcv[it, isplit] = np.mean((yhat-yts)**2)

One approach is the nested for-loop approach:
1. Loop over test fold $$k = 1, \dots, K$$
2. Loop over model orders $$d = 1, \dots, D$$
3. Compute $$sMSE$$ $$d$$, $$k$$ for each d and k combination
4. Average $$sMSE$$ across test folds to get $$\overline{sMSE}_{d}\triangleq \frac{1}{K}\sum_{k=1}^{K}sMSE_{d,k}$$
5. Choose the $$d$$ giving the smallest $$\overline{sMSE}_{d}$$

But this imperfeclty estimates the $$MSE_{d}$$. To overcome this we can do:

In [ ]:
sMSE_mean = np.mean(sMSEcv, axis=1)
sMSE_se = np.std(sMSEcv, axis=1, ddof=1)/np.sqrt(k)
plt.errorbar(d_test, sMSE_mean, yerr=sMSE_se, fmt='-')
plt.ylim(0,1.5)
plt.xlabel("Model Order")
plt.ylabel("mean CV-sMSE")
plt.grid()

**Even though** we did say that choosing the d that minimizes $$\overline{sMSE}_{d}$$, this can still *sometimes* lead to choosing an overfit model. So to overcome this we can choose the simplest model that gives performance within one SE of the best model.

To do this you can:
* Find the best model that has the lowest $$\overline{sMSE}_{d}$$
* Find the $$\overline{sMSE}_{target}$$ where $$\overline{sMSE}_{target}= \overline{sMSE}_{lowest} + SE$$
* Resulting in find the smallest $$d$$ where $$\overline{sMSE}_{d} \leq \overline{sMSE}_{target}$$

In [ ]:
min_sMSE = np.argmin(sMSE_mean)
tgt_sMSE = sMSE_mean[min_sMSE] + sMSE_se[min_sMSE]
mask = np.where(sMSE_mean <= tgt_sMSE)

print("lowest sMSE at d:", np.argmin(sMSE_mean))
print("lowest one standard error d:", np.min(mask))

### What do we need to know?
With polynomial regression:
* choosing $$d$$ too small can cause underfitting
* choosing $$d$$ too big can cause overfitting

But you can choose an optimal $$d$$ by minimizing the sample $$sMSE_{test}$$ through cross-validation.

But the key takeaway is that of a more general concept:
* models that are too simple cause underfitting
* models that are too complex cause overfitting

#### But complexity can be optimized by minimizing an appropriate metric through cross-validation.